# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [10]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [11]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


Getting mental health therapy involves a few structured steps, from identifying your needs to selecting a specific treatment and finding a qualified provider. Here is a comprehensive guide on how to navigate the process, the therapy options available, and the resources you can use, based on current healthcare guidelines.

### 1. Identify Your Needs, Symptoms, and Goals
Before searching for a therapist, clarify your goals and needs [1]. Think about the specific symptoms or conditions you are dealing with, as different therapies are tailored to treat different issues. Mental health therapy is routinely used to address and treat:
*   Anxiety and depression [2]
*   Trauma and PTSD [3]
*   Eating disorders (which require specialized identification and available treatments) [4]
*   Substance and alcohol use disorders [4]
*   Chronic pain, chronic illness (like COPD or heart conditions), and life transitions that impact mental wellness [4]

### 2. Understand Your Therapy Options and Treatments
Therapy is not one-size-fits-all, and different formats and procedures work for different people. A qualified provider will typically start with a full clinical assessment and work with you to create a personalized treatment plan [2]. Treatment options can stimulate healthy changes in the brain, improve emotional control, and help you process feelings [5]. 

**Common Types of Therapy:**
*   **Cognitive Behavioral Therapy (CBT):** An action-oriented approach that explores the link between thoughts, feelings, and behaviors to change negative patterns [3][5][6].
*   **Dialectical Behavior Therapy (DBT):** Focuses on teaching emotional regulation and distress tolerance skills [5][6].
*   **Acceptance and Commitment Therapy (ACT):** Utilizes mindfulness (experiencing things without judgment) and behavior change to cope with unwanted thoughts and feelings [2][5].
*   **Eye Movement Desensitization and Reprocessing (EMDR):** Specifically designed and highly effective for treating trauma and PTSD [3].
*   **Psychodynamic & Humanistic Therapies:** These focus on self-reflection, understanding unconscious past influences, and facilitating personal growth [3].
*   **Interpersonal Therapy (IPT) & Mindfulness-based cognitive therapy:** Focus on improving relationships and incorporating mindfulness into daily routines [3].

**Therapy Formats:**
You will also need to decide if you prefer individual (one-on-one) sessions, couples and family therapy, group therapy, or online/telehealth therapy options [3][5]. Additionally, psychotherapy is sometimes combined with medication prescribed by a psychiatrist depending on your condition [6].

### 3. Use Resources to Find a Therapist
Once you know what you are looking for, you can start looking for a licensed provider. The best places to look include:
*   **Insurance Directories:** Call the number on the back of your insurance card or log into your insurance portal to identify in-network providers, which will make therapy more affordable [7][8].
*   **Primary Care Referrals:** Ask your primary care physician, nurse practitioner, or another medical professional for a referral. You can also ask trusted friends or family members [7][9].
*   **Online Therapy Platforms and Directories:** You can search for ethical, licensed, and qualified therapists using dedicated mental health directories. Reputable ones include:
    *   *GoodTherapy* directory [10]
    *   *Anxiety & Depression Association of America (ADAA)* "Find Your Therapist" tool [2]
    *   *Mental Health America (MHA)* resources [6]
    *   *Spring Health* or similar care navigation platforms [7]

### 4. Contact, Interview, and Assess Potential Therapists
Finding a therapist is a highly personal process, so it is recommended to "do your homework" and ensure the professional is a good fit [11]. 
*   **Check Credentials:** Ensure they are licensed in your state and have experience treating your specific symptoms [1][12]. 
*   **Ask Questions:** Call potential therapists for a consultation. The American Psychological Association recommends asking if they are familiar with "evidence-based treatments" for your specific concerns and if they actively use those treatments in their practice [12]. 
*   **Consider Logistics and Comfort:** Take into account location, pricing, availability, and most importantly, your gut feeling. A strong therapeutic relationship is crucial for success [1][11]. 

*Note: Treatment results vary; some people feel relief within a few weeks, while those dealing with multiple or complex conditions may need more time to see improvement [2].*

***

**Sources:**
[1] Comprehensive Healthcare (comphc.org) - *Finding the Right Therapist: A Step-by-Step Guide*
[2] Anxiety and Depression Association of America (adaa.org) - *Mental Health Therapy Types*
[3] HelpGuide (helpguide.org) - *Therapy Types* 
[4] Highmark Health (highmarkhealth.org) - *Ask a Doc: Choosing Mental Health Therapy & Care Connections*
[5] GoodRx (goodrx.com) - *Mental Health Therapy Options*
[6] Mental Health America (mhanational.org) - *Mental Health Treatments*
[7] Spring Health (springhealth.com) - *Where is the best place to find a therapist?*
[8] IRTBH (irtbh.com) - *How to Get Mental Health Therapy Covered by Insurance*
[9] Therapists in Charlotte (therapistsincharlotte.com) - *Referrals for Therapy*
[10] GoodTherapy (goodtherapy.org) - *Find a Therapist*
[11] Sharp HealthCare (sharpmedicareadvantage.com) - *Tips for Finding a Therapist*
[12] American Psychological Association (apa.org) - *How Do I Find a Good Therapist?*

Based on the insurance document provided, here's what you need to know about mental health therapy coverage under the Anthem Blue Cross gHIP HDHP plan:

## Mental Health Therapy Coverage

**Outpatient Services (Office Visits):**
- **In-Network Provider:** 10% coinsurance
- **Out-of-Network Provider:** 30% coinsurance

**Other Outpatient Services:**
- **In-Network Provider:** 10% coinsurance
- **Out-of-Network Provider:** 30% coinsurance

**Inpatient Services:**
- **In-Network Provider:** 10% coinsurance (plus 10% coinsurance for inpatient physician fees)
- **Out-of-Network Provider:** 30% coinsurance (plus 30% coinsurance for inpatient physician fees)

## Important Coverage Details

1. **Deductible Applies:** These coinsurance costs apply after your deductible has been met (\\$1,700 individual/\\$3,400 family for in-network providers).

2. **No Referral Required:** You can see a mental health specialist without needing a referral.

3. **Network Providers Available:** 
   - NY Blue Access PPO (if in NY, excluding Suffolk County)
   - GA Blue Open Access POS (if in GA)
   - Blue Card PPO (all other areas)
   - Visit **includedhealth.com/google** or call **(855) 431-5540** for a list of in-network mental health providers.

4. **Cost Savings:** You'll pay significantly less by using in-network providers (10% vs. 30% coinsurance).

For more detailed information about specific limitations or coverage details, contact the plan at **(855) 431-5540** or review your complete Benefits Booklet.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
